# Stacked and bidirectional recurrence

**Learning objective:** Build deeper and bidirectional recurrent networks and understand when they are valid.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:13:50.194869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974830.210165    3138 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974830.214651    3138 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:13:51.872597: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
stacked=tf.keras.Sequential([
    tf.keras.layers.Input((20,3)),
    tf.keras.layers.GRU(12,return_sequences=True),
    tf.keras.layers.GRU(8),
    tf.keras.layers.Dense(1)
],name="stacked_gru")
bidir=tf.keras.Sequential([
    tf.keras.layers.Input((20,3)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(8)),
    tf.keras.layers.Dense(1)
],name="bidirectional_lstm")
print("stacked output:",stacked(tf.zeros((4,20,3))).shape,"parameters:",stacked.count_params())
print("bidirectional output:",bidir(tf.zeros((4,20,3))).shape,"parameters:",bidir.count_params())


stacked output: (4, 1) parameters: 1149
bidirectional output: (4, 1) parameters: 785


**Critical causality rule:** bidirectional recurrence uses future context. It is excellent when the entire sequence is available at inference (e.g., document classification) but invalid for causal forecasting where future timesteps do not yet exist.


In [3]:
display(pd.DataFrame({"use case":["offline text classification","causal sensor forecast"],"bidirectional valid?":[True,False],"reason":["whole sequence known","future context would leak"]}))


,use case,bidirectional valid?,reason
0,offline text classification,True,whole sequence known
1,causal sensor forecast,False,future context would leak
